In [12]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.messages import HumanMessage, SystemMessage
import base64
load_dotenv(".env.txt")
import json

In [13]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)

agent = create_agent(tools=[],model=llm,
                      system_prompt=())


In [2]:
def encode_img_to_data_url(path: Path)-> str:

    """convert a local image file to a data URL usable in image_url inputs. """

    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

In [6]:
base_dir = Path.cwd()

img1 = encode_img_to_data_url(base_dir/"images"/"demo.jpg")
img2 = encode_img_to_data_url(base_dir/"images"/"demo1.jpg")
img3 = encode_img_to_data_url(base_dir/"images"/"demo2.jpg")


In [9]:
def get_item_code(item_name: str) -> str:
    if item_name == "sari":
        return "ITM001"
    if item_name == "t-shirt":
        return "ITM002"
    if item_name == "jeans":
        return "ITM003"
    if item_name == "jacket":
        return "ITM004"
    return "ITM999"

In [ ]:
SYSTEM_PROMPT = """For each image return a JSON array of records. Each record must have:
    - item_name: one of sari, t-shirt, jeans, jacket
    - color: the dominant color of the clothing item
    - gender: male, female, or unisex
    - age_category: adult or child

Output only a raw JSON array. No preamble, no markdown, no code fences."""

In [ ]:
human_content = [
    {"type": "text", "text": "Analyze each image and return a JSON array of records as instructed."},
    {"type": "image_url", "image_url": {"url": img1}},
    {"type": "image_url", "image_url": {"url": img2}},
    {"type": "image_url", "image_url": {"url": img3}},
]

In [ ]:
response = llm.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=human_content),
])

records = json.loads(response.content)

for record in records:
    record["item_code"] = get_item_code(record["item_name"])

print(json.dumps(records, indent=2))